In [ ]:
# ── All imports ───────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# ML libraries — these come pre-installed in Colab
from sklearn.model_selection  import train_test_split
from sklearn.preprocessing    import StandardScaler
from sklearn.linear_model     import LogisticRegression
from sklearn.ensemble         import RandomForestClassifier
from sklearn.cluster          import KMeans
from sklearn.metrics          import (
    roc_auc_score, classification_report,
    confusion_matrix, roc_curve, ConfusionMatrixDisplay
)
try:
    from xgboost import XGBClassifier
    print('✓ XGBoost available')
except ImportError:
    import subprocess
    subprocess.run(['pip', 'install', 'xgboost', '-q'])
    from xgboost import XGBClassifier
    print('✓ XGBoost installed and loaded')



In [ ]:

df = pd.read_csv('06_feature_table.csv')

print(f'✓ Feature table loaded: {df.shape[0]:,} rows × {df.shape[1]} cols')
print(f'\nDefault rate: {df["default_flag"].mean()*100:.1f}%')
print(f'Defaulted loans  : {df["default_flag"].sum():,}')
print(f'Non-default loans: {(df["default_flag"]==0).sum():,}')
print(f'\nNull values: {df.isnull().sum().sum()}')

In [ ]:


FEATURES = [
    # Affordability
    'income_to_emi', 'emi_to_income_pct', 'total_debt_burden_pct', 'loan_to_income_ratio',
    # Credit quality
    'credit_bureau_score', 'thin_file_flag', 'credit_history_score',
    'risk_grade_numeric', 'years_credit_history',
    # Product
    'loan_amount', 'tenure_months', 'interest_rate_pct',
    'rate_spread', 'long_tenure_flag', 'product_risk_score',
    # Acquisition
    'channel_risk_score', 'cac_to_loan_ratio', 'approval_turnaround_days',
    # Demographics
    'age', 'employment_risk_score', 'city_tier_risk',
    # Repayment behavior
    'payment_ratio', 'dpd_max', 'dpd_avg', 'delinquency_rate',
    'missed_emi_rate', 'partial_payment_rate', 'dpd_trend_slope', 'stress_score',
    # Behavioral
    'avg_balance', 'avg_volatility', 'avg_transactions',
    'spending_shock_count', 'avg_cashflow_score',
    'balance_stability_ratio', 'shock_frequency', 'cashflow_stress_flag',
    'balance_to_emi'
]

TARGET = 'default_flag'

X = df[FEATURES]
y = df[TARGET]

print(f'✓ Features selected: {len(FEATURES)}')
print(f'  X shape: {X.shape}')
print(f'  y shape: {y.shape}')

In [ ]:


X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size   = 0.20,
    random_state= 42,
    stratify    = y
)

print(f'Training set   : {X_train.shape[0]:,} rows')
print(f'Test set       : {X_test.shape[0]:,} rows')
print(f'\nDefault rate in train: {y_train.mean()*100:.1f}%')
print(f'Default rate in test : {y_test.mean()*100:.1f}%')
print('✓ Rates are same — stratify worked!')

In [ ]:


scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print('✓ Scaling done')
print(f'  X_train_scaled shape: {X_train_scaled.shape}')
print(f'  X_test_scaled shape : {X_test_scaled.shape}')

In [ ]:

print('Training Logistic Regression...')

lr_model = LogisticRegression(
    max_iter   = 1000,
    random_state = 42,
    class_weight = 'balanced'
)

lr_model.fit(X_train_scaled, y_train)

lr_probs = lr_model.predict_proba(X_test_scaled)[:, 1]
lr_preds = lr_model.predict(X_test_scaled)

lr_auc = roc_auc_score(y_test, lr_probs)
print(f'  AUC-ROC Score: {lr_auc:.4f}')
print(classification_report(y_test, lr_preds, target_names=['No Default','Default']))

In [ ]:


fig, ax = plt.subplots(figsize=(6, 5))
cm = confusion_matrix(y_test, lr_preds)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['No Default','Default'])
disp.plot(ax=ax, colorbar=False, cmap='Blues')
ax.set_title(f'Logistic Regression — Confusion Matrix\nAUC: {lr_auc:.3f}', fontweight='bold')
plt.tight_layout()
plt.show()

tn, fp, fn, tp = cm.ravel()
print(f'\nTrue Negatives  (Correct no-default): {tn:,}')
print(f'True Positives  (Correct default)   : {tp:,}')
print(f'False Positives (Wrong alarm)        : {fp:,}')
print(f'False Negatives (MISSED defaults)    : {fn:,}  ← most dangerous')

In [ ]:


rf_model = RandomForestClassifier(
    n_estimators  = 100,
    max_depth     = 8,    
    min_samples_leaf = 20,  
    class_weight  = 'balanced', 
    random_state  = 42,
    n_jobs        = -1 
)

rf_model.fit(X_train, y_train)

rf_probs = rf_model.predict_proba(X_test)[:, 1]
rf_preds = rf_model.predict(X_test)
rf_auc   = roc_auc_score(y_test, rf_probs)

print(f'  AUC-ROC Score: {rf_auc:.4f}')

print(classification_report(y_test, rf_preds, target_names=['No Default','Default']))

In [ ]:

feat_imp = pd.DataFrame({
    'feature'   : FEATURES,
    'importance': rf_model.feature_importances_
}).sort_values('importance', ascending=False).head(15)

fig, ax = plt.subplots(figsize=(9, 7))
ax.barh(feat_imp['feature'][::-1], feat_imp['importance'][::-1],
         edgecolor='white')
ax.set_title('Random Forest — Top 15 Feature Importances', fontsize=13, fontweight='bold')
ax.set_xlabel('Importance Score')
plt.tight_layout()
plt.show()

for i, row in feat_imp.head(5).iterrows():
    print(f'   {row["feature"]:<35} {row["importance"]:.4f}')

In [ ]:

neg_count = (y_train == 0).sum()
pos_count = (y_train == 1).sum()
scale_weight = neg_count / pos_count
print(f'  scale_pos_weight = {scale_weight:.1f} (default cases ko {scale_weight:.0f}x weight)')

xgb_model = XGBClassifier(
    n_estimators      = 200,  
    max_depth         = 5,           
    learning_rate     = 0.05,         
    scale_pos_weight  = scale_weight, 
    random_state      = 42,
    eval_metric       = 'auc',
    verbosity         = 0             
)

xgb_model.fit(
    X_train, y_train,
    eval_set        = [(X_test, y_test)],
    verbose         = False
)

xgb_probs = xgb_model.predict_proba(X_test)[:, 1]
xgb_preds = xgb_model.predict(X_test)
xgb_auc   = roc_auc_score(y_test, xgb_probs)

print(f'  AUC-ROC Score: {xgb_auc:.4f}')

print(classification_report(y_test, xgb_preds, target_names=['No Default','Default']))

In [ ]:

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for probs, label, color in [
    (lr_probs,  f'Logistic Regression (AUC={lr_auc:.3f})',  BLUE),
    (rf_probs,  f'Random Forest       (AUC={rf_auc:.3f})',  GREEN),
    (xgb_probs, f'XGBoost             (AUC={xgb_auc:.3f})', RED)
]:
    fpr, tpr, _ = roc_curve(y_test, probs)
    axes[0].plot(fpr, tpr, label=label, linewidth=2, color=color)

axes[0].plot([0,1],[0,1], 'k--', linewidth=1, label='Random (AUC=0.5)')
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('ROC Curves — Model Comparison', fontweight='bold')
axes[0].legend(fontsize=9)

models     = ['Logistic\nRegression', 'Random\nForest', 'XGBoost']
auc_scores = [lr_auc, rf_auc, xgb_auc]
bars = axes[1].bar(models, auc_scores,edgecolor='white', width=0.4)
for bar, val in zip(bars, auc_scores):
    axes[1].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.002,
                 f'{val:.4f}', ha='center', fontweight='bold')
axes[1].set_ylim(0.5, 1.0)
axes[1].set_ylabel('AUC-ROC Score')
axes[1].set_title('AUC Comparison', fontweight='bold')
axes[1].axhline(0.8, linestyle='--', linewidth=1.5, label='Good threshold (0.80)')
axes[1].legend()

plt.tight_layout()
plt.show()
best_auc   = max(lr_auc, rf_auc, xgb_auc)
best_model = ['Logistic Regression','Random Forest','XGBoost'][auc_scores.index(best_auc)]
print(f'\n {best_model} with AUC = {best_auc:.4f}')


In [ ]:

df['default_probability'] = xgb_model.predict_proba(X)[:, 1].round(4)
df['risk_category'] = pd.cut(
    df['default_probability'],
    bins   = [0, 0.15, 0.35, 0.60, 1.01],
    labels = ['Low Risk', 'Medium Risk', 'High Risk', 'Critical Risk']
)

print('Risk Category Distribution:')
risk_dist = df['risk_category'].value_counts().sort_index()
for cat, count in risk_dist.items():
    pct = count/len(df)*100
    bar = '█' * int(pct/2)
    print(f'  {cat:<15} {count:>5,} ({pct:4.1f}%)  {bar}')

for cat in ['Low Risk','Medium Risk','High Risk','Critical Risk']:
    sub = df[df['risk_category']==cat]
    dr  = sub['default_flag'].mean()*100 if len(sub) > 0 else 0
    print(f'  {cat:<15} {dr:.1f}% actual default rate')

##Early Warning System — Model 2

In [ ]:


EARLY_FEATURES = [
    # Application time features (known before loan starts)
    'income_to_emi', 'credit_bureau_score', 'thin_file_flag',
    'risk_grade_numeric', 'loan_amount', 'product_risk_score',
    'channel_risk_score', 'employment_risk_score', 'city_tier_risk',
    'age', 'years_credit_history', 'total_debt_burden_pct',
    # Month 1-3 repayment signals
    'early_delinquency_flag', 'early_stress_flag', 'early_max_dpd',
    # Behavioral signals (available from bank data)
    'avg_volatility', 'spending_shock_count', 'avg_cashflow_score',
    'balance_stability_ratio', 'cashflow_stress_flag'
]

X_early = df[EARLY_FEATURES]
y_early = df[TARGET]

X_train_e, X_test_e, y_train_e, y_test_e = train_test_split(
    X_early, y_early, test_size=0.20, random_state=42, stratify=y_early
)

# Same XGBoost — same settings
ews_model = XGBClassifier(
    n_estimators     = 200,
    max_depth        = 5,
    learning_rate    = 0.05,
    scale_pos_weight = scale_weight,
    random_state     = 42,
    eval_metric      = 'auc',
    verbosity        = 0
)
ews_model.fit(X_train_e, y_train_e, verbose=False)

ews_probs = ews_model.predict_proba(X_test_e)[:, 1]
ews_auc   = roc_auc_score(y_test_e, ews_probs)

print(f'\n  Full model AUC  (all features)     : {xgb_auc:.4f}')
print(f'  Early Warning AUC (months 1-3 only): {ews_auc:.4f}')

In [ ]:
# ── Early Warning — how many defaults can we catch early? ─
threshold = 0.30 

ews_flags = (ews_probs >= threshold).astype(int)
actual    = y_test_e.values

actual_defaults      = actual.sum()
flagged_correctly    = ((ews_flags == 1) & (actual == 1)).sum()
flagged_incorrectly  = ((ews_flags == 1) & (actual == 0)).sum()
missed_defaults      = ((ews_flags == 0) & (actual == 1)).sum()

recall_pct = flagged_correctly / actual_defaults * 100


## 7 · Model 3 — Customer Segmentation (KMeans Clustering)
>

In [ ]:

CLUSTER_FEATURES = [
    'credit_bureau_score',   # creditworthiness
    'income_to_emi',         # affordability
    'stress_score',          # repayment stress
    'avg_volatility',        # financial stability
    'risk_grade_numeric',    # lender's own risk assessment
    'employment_risk_score', # income stability
    'dpd_max',               # worst delinquency
    'payment_ratio',         # overall payment discipline
]

X_cluster = df[CLUSTER_FEATURES].copy()
scaler_c        = StandardScaler()
X_cluster_scaled= scaler_c.fit_transform(X_cluster)

print(f'✓ Clustering features selected: {len(CLUSTER_FEATURES)}')

In [ ]:

inertia_values = []
k_range        = range(2, 9)

for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_cluster_scaled)
    inertia_values.append(km.inertia_)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(k_range, inertia_values, marker='o', linewidth=2, markersize=8)
ax.set_xlabel('Number of Clusters (k)')
ax.set_ylabel('Inertia')
ax.set_title('Elbow Method — Optimal Number of Clusters', fontweight='bold')

ax.axvline(4, linestyle='--', linewidth=1.5, label='Elbow at k=4')
ax.legend()
plt.tight_layout()
plt.show()


In [ ]:

K = 4 

kmeans = KMeans(n_clusters=K, random_state=42, n_init=10)
df['segment'] = kmeans.fit_predict(X_cluster_scaled)

print(f'✓ KMeans clustering done with k={K}')
for seg, count in df['segment'].value_counts().sort_index().items():
    print(f'  Segment {seg}: {count:,} loans ({count/len(df)*100:.1f}%)')

In [ ]:

segment_profile = df.groupby('segment').agg(
    count             = ('loan_id','count'),
    default_rate      = ('default_flag','mean'),
    avg_credit_score  = ('credit_bureau_score','mean'),
    avg_income_to_emi = ('income_to_emi','mean'),
    avg_stress_score  = ('stress_score','mean'),
    avg_volatility    = ('avg_volatility','mean'),
    avg_dpd_max       = ('dpd_max','mean'),
    avg_payment_ratio = ('payment_ratio','mean'),
    pct_salaried      = ('employment_risk_score', lambda x: (x==1).mean()),
    pct_gig           = ('employment_risk_score', lambda x: (x==4).mean()),
).round(3)
segment_profile['default_rate'] = (segment_profile['default_rate'] * 100).round(1)

print('Segment Profiles:')
print(segment_profile.to_string())

In [ ]:

sorted_segs = segment_profile['default_rate'].sort_values()
print('Segments sorted by default rate (low to high):')
for seg, dr in sorted_segs.items():
    cs  = segment_profile.loc[seg, 'avg_credit_score']
    ss  = segment_profile.loc[seg, 'avg_stress_score']
    cnt = segment_profile.loc[seg, 'count']
    print(f'  Segment {seg}: Default={dr:.1f}% | Credit Score={cs:.0f} | Stress={ss:.3f} | Count={cnt:,}')
seg_order = sorted_segs.index.tolist()
seg_names = {
    seg_order[0]: 'Prime Borrowers',       
    seg_order[1]: 'Emerging Borrowers',    
    seg_order[2]: 'Stressed Borrowers',    
    seg_order[3]: 'High Risk Borrowers'   
}

df['segment_name'] = df['segment'].map(seg_names)
print('\n✓ Segment names assigned:')
for k,v in seg_names.items():
    print(f'   Segment {k} → {v}')

In [ ]:

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Customer Segments — Risk Profile Comparison', fontsize=13, fontweight='bold')
seg_display = df.groupby('segment_name').agg(
    default_rate     =('default_flag','mean'),
    avg_credit_score =('credit_bureau_score','mean'),
    avg_stress_score =('stress_score','mean'),
    count            =('loan_id','count')
).reset_index()
seg_display['default_rate'] *= 100
seg_display = seg_display.sort_values('default_rate')

# Default rate
bars = axes[0].bar(seg_display['segment_name'], seg_display['default_rate'],
                    edgecolor='white')
for bar, val in zip(bars, seg_display['default_rate']):
    axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.3,
                 f'{val:.1f}%', ha='center', fontweight='bold', fontsize=9)
axes[0].set_title('Default Rate by Segment')
axes[0].set_ylabel('Default Rate (%)')
axes[0].tick_params(axis='x', rotation=15)

# Credit score
axes[1].bar(seg_display['segment_name'], seg_display['avg_credit_score'],
             edgecolor='white')
axes[1].set_title('Avg Credit Score')
axes[1].set_ylabel('Credit Score')
axes[1].tick_params(axis='x', rotation=15)

# Segment size
axes[2].pie(seg_display['count'], labels=seg_display['segment_name'],
            autopct='%1.1f%%', startangle=90)
axes[2].set_title('Segment Size Distribution')

plt.tight_layout()
plt.show()

In [ ]:

policies = {
    'Prime Borrowers': {
        'action'   : 'REWARD & GROW',
        'policy'   : 'Offer higher loan limits, lower interest rates, pre-approved top-ups',
        'pricing'  : 'Rate: Grade A pricing (11-14%)',
        'risk'     : 'Low — monitor quarterly'
    },
    'Emerging Borrowers': {
        'action'   : 'NURTURE & MONITOR',
        'policy'   : 'Standard approval, shorter initial tenure, review after 6 EMIs',
        'pricing'  : 'Rate: Grade B pricing (15-19%)',
        'risk'     : 'Medium — monthly DPD monitoring'
    },
    'Stressed Borrowers': {
        'action'   : 'RESTRICT & SUPPORT',
        'policy'   : 'Lower ticket size cap, mandatory income verification, early collections alert',
        'pricing'  : 'Rate: Grade C pricing (20-26%)',
        'risk'     : 'High — trigger early warning at first DPD event'
    },
    'High Risk Borrowers': {
        'action'   : 'DECLINE OR SECURED ONLY',
        'policy'   : 'Decline unsecured loans, offer only secured/co-signed products',
        'pricing'  : 'Rate: Grade D pricing (27-36%) or decline',
        'risk'     : 'Critical — immediate collections if any DPD'
    }
}

for seg_name, pol in policies.items():
    seg_data = seg_display[seg_display['segment_name'] == seg_name]
    if len(seg_data) > 0:
        count = seg_data['count'].values[0]
        dr    = seg_data['default_rate'].values[0]
        print(f'\n  [{seg_name}] — {count:,} loans | Default: {dr:.1f}%')
        print(f'    Action  : {pol["action"]}')
        print(f'    Policy  : {pol["policy"]}')
        print(f'    Pricing : {pol["pricing"]}')
        print(f'    Risk mgmt: {pol["risk"]}')


In [ ]:
final_output = df[[
    'loan_id', 'customer_id', 'product_type', 'risk_grade',
    'loan_amount', 'tenure_months', 'acquisition_channel',
    'default_flag', 'loan_status',
    'default_probability', 
    'risk_category',         
    'segment_name',          
    'stress_score',          
    'early_stress_flag',   
    'income_to_emi',
    'credit_bureau_score',
    'dpd_max',
    'payment_ratio'
]]

final_output.to_csv('07_scored_loans.csv', index=False)